# TealKit MCP Agent — Weather Sensors MCP — Ministral-3B Training Pipeline

Fine-tunes **Ministral-3B** (`unsloth/Ministral-3-3B-Instruct-2512`) for Weather Sensors MCP tool-calling via Unsloth LoRA on Colab H100/L4.

**Why Ministral-3B?**
Ministral 3B (December 2025) is a Mistral-family 3B model explicitly designed for edge function-calling.
Unlike Qwen3-4B it has no thinking mode, so `tool_call:` format stays clean after fine-tuning.

**Key Features:**
1. **Mistral Chat Template:** Uses the native Mistral `[INST]...[/INST]` chat format. No thinking-mode suppression needed.
2. **Loss-Masking:** Masks user prompt tokens so the model learns only assistant/tool-call responses.
3. **Memory-Cleared Export:** Drops the trained model from GPU memory before GGUF export to prevent Colab OOM crashes.
4. **Same Tool-Call Format:** Outputs `tool_call: {"name":"...","arguments":{...}}` — identical to the Qwen2.5-3B Mac path.

## Cell 1 — Install Dependencies (Fixed Cache)
> This installs the very latest Unsloth and wipes the broken `llama.cpp` cache to prevent the 'conversion' module error.

In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" trl peft accelerate bitsandbytes datasets huggingface_hub

import shutil
shutil.rmtree('/root/.unsloth', ignore_errors=True)

print('Install done. Restarting runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 2 — ⚙️ Config

In [ ]:
MODEL_NAME = 'unsloth/Ministral-3-3B-Instruct-2512'

# --- Update these paths to match your Google Drive layout ---
# If you generated data with run_generate_weather_train_jsonl_ministral.sh,
# upload mcp_out_ministral/ to Drive.  Otherwise point at your existing mcp_out/ folder.
DATA_DIR = '/content/drive/MyDrive/Tealkit/training/weathersensorsmcp/mcp_out_ministral'
OUTPUT_DIR = '/content/drive/MyDrive/Tealkit/training/weathersensorsmcp/mcp_adapters_ministral_3b'
GGUF_DIR = '/content/drive/MyDrive/Tealkit/training/weathersensorsmcp/mcp_fused_model_ministral_3b'
MERGE_DIR = '/content/drive/MyDrive/Tealkit/training/weathersensorsmcp/mcp_merged_model_ministral_3b'
SYSTEM_PROMPT_FILE = '/content/drive/MyDrive/Tealkit/training/weathersensorsmcp/weather_sensors_system_prompt.md'
PREFER_EMBEDDED_UPDATED_PROMPT = True

# Hugging Face repo for upload (Cell 10)
HF_REPO = 'lschaffer/ministral-3b-weathersensorsmcp'  # <-- Change to your HF username/repo

# Training settings
MAX_SEQ_LENGTH = 4096

TRAIN_FILE = f'{DATA_DIR}/train_split.jsonl'
VALID_FILE = f'{DATA_DIR}/valid_split.jsonl'

print('Model                 :', MODEL_NAME)
print('Train file            :', TRAIN_FILE)
print('Valid file            :', VALID_FILE)
print('Drive system prompt   :', SYSTEM_PROMPT_FILE)
print('Prefer embedded prompt:', PREFER_EMBEDDED_UPDATED_PROMPT)
print('Adapters out          :', OUTPUT_DIR)
print('GGUF out              :', GGUF_DIR)
print('HF repo               :', HF_REPO)

## Cell 3 — Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

for _path, _label in [(TRAIN_FILE, 'train_split.jsonl'), (VALID_FILE, 'valid_split.jsonl'), (SYSTEM_PROMPT_FILE, 'weather system prompt')]:
    if not os.path.isfile(_path):
        print(f'MISSING {_label}: {_path} -> Upload to Drive first!')
    else:
        print(f'OK  {_label} found at {_path}')

## Cell 4 — Load model
> Ministral 3B in native bfloat16. No thinking-mode suppression needed.

In [ ]:
from unsloth import FastLanguageModel
import torch
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='mistral',
)

print('Model loaded in native bfloat16 precision.')

## Cell 5 — Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## Cell 6 — Load dataset + text-contract prompt
> Uses the Mistral chat template and trains the legacy `tool_call: {...}` contract.

In [ ]:
from datasets import load_dataset
import json
import os

UPDATED_SYSTEM_PROMPT = '''You are the Cumulus Assistant for Weather Sensors MCP.

When a tool is needed, respond only with:
tool_call: {"name":"<tool_name>","arguments":{...}}

After tool results are available, answer only with the requested result. Do not add suggestions or follow-up offers.'''

if os.path.isfile(SYSTEM_PROMPT_FILE):
    with open(SYSTEM_PROMPT_FILE, 'r', encoding='utf-8') as handle:
        SYSTEM_PROMPT = handle.read().strip()
else:
    SYSTEM_PROMPT = UPDATED_SYSTEM_PROMPT

def format_example(examples):
    texts = []
    for msgs in examples['messages']:
        if not msgs or msgs[0].get('role') != 'system':
            msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}] + list(msgs)
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {'text': texts}

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'validation': VALID_FILE})
dataset = dataset.map(format_example, batched=True)

def detect_chat_markers(sample_text):
    instruction_candidates = ['[INST] ', '<|im_start|>user\n', '<|im_start|>user<|im_sep|>', '<|user|>']
    response_candidates = [' [/INST]', '<|im_start|>assistant\n', '<|im_start|>assistant<|im_sep|>', '<|assistant|>']
    instruction_part = next((part for part in instruction_candidates if part in sample_text), None)
    response_part = next((part for part in response_candidates if part in sample_text), None)
    return instruction_part, response_part

sample_text = dataset['train'][0]['text'] if len(dataset['train']) else ''
INSTRUCTION_PART, RESPONSE_PART = detect_chat_markers(sample_text)
print('Train examples:', len(dataset['train']))
print('Valid examples:', len(dataset['validation']))
print('Instruction marker:', repr(INSTRUCTION_PART))
print('Response marker   :', repr(RESPONSE_PART))
print(sample_text[:2000])

## Cell 7 — Training Loop

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

trainer_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    output_dir='/content/outputs',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=trainer_args,
)

if INSTRUCTION_PART and RESPONSE_PART:
    try:
        masked_trainer = train_on_responses_only(
            trainer,
            instruction_part=INSTRUCTION_PART,
            response_part=RESPONSE_PART,
        )
        if len(masked_trainer.train_dataset) == 0:
            print('WARNING: Response masking removed every train sample. Falling back to full-sequence training.')
        else:
            trainer = masked_trainer
            print(f'Response masking active: {len(trainer.train_dataset)} train samples.')
    except Exception as exc:
        print('WARNING: Response masking failed, using full sequence. Error:', exc)
else:
    print('WARNING: Could not detect chat markers in formatted text. Using full-sequence training.')

trainer.train()
print('Training complete.')

## Cell 8 — Save adapters + export GGUF

In [ ]:
import gc
import glob
import os
import shutil
import subprocess
import sys

QUANT_METHOD = 'q5_k_m'
GGUF_BASENAME = f"{HF_REPO.split('/')[-1]}-unsloth"
GGUF_F16_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-F16.gguf')
GGUF_QUANT_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-{QUANT_METHOD.upper()}.gguf')
FINAL_GGUF_FILE = None
GGUF_FILENAME = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapters successfully saved to Drive:', OUTPUT_DIR)

def run_checked(command, cwd=None, extra_env=None):
    env = os.environ.copy()
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env['PYTHONNOUSERSITE'] = '1'
    if extra_env:
        for key, value in extra_env.items():
            if value is None:
                env.pop(key, None)
            else:
                env[key] = value
    print('>>', ' '.join(command))
    try:
        subprocess.run(command, cwd=cwd, env=env, check=True)
    except subprocess.CalledProcessError:
        # Re-run with stderr captured to show the actual error
        result = subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)
        err = result.stderr
        if len(err) > 3000:
            print('=== STDERR (last 3000 chars) ===')
            print(err[-3000:])
        else:
            print('=== STDERR ===')
            print(err)
        raise

def clear_unsloth_llama_cpp_cache():
    cache_dir = '/root/.unsloth/llama.cpp'
    if os.path.isdir(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)
        print('Cleared cached Unsloth llama.cpp checkout:', cache_dir)

def refresh_tokenizer_files(merged_dir, base_model):
    from huggingface_hub import hf_hub_download
    for filename in ('tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json'):
        try:
            source_path = hf_hub_download(repo_id=base_model, filename=filename)
            shutil.copy2(source_path, os.path.join(merged_dir, filename))
        except Exception as exc:
            print(f'INFO: Could not refresh {filename}: {exc}')
    try:
        source_path = hf_hub_download(repo_id=base_model, filename='tokenizer.model')
        shutil.copy2(source_path, os.path.join(merged_dir, 'tokenizer.model'))
    except Exception:
        pass

def ensure_llama_cpp_checkout(llama_cpp_dir):
    convert_script = os.path.join(llama_cpp_dir, 'convert_hf_to_gguf.py')
    if not os.path.isdir(os.path.join(llama_cpp_dir, '.git')):
        if os.path.exists(llama_cpp_dir):
            shutil.rmtree(llama_cpp_dir, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', llama_cpp_dir])
    return convert_script

def manual_llama_cpp_convert(merged_dir):
    clear_unsloth_llama_cpp_cache()
    refresh_tokenizer_files(merged_dir, MODEL_NAME)
    llama_cpp_dir = '/content/llama.cpp'
    convert_script = ensure_llama_cpp_checkout(llama_cpp_dir)
    gguf_py_dir = os.path.join(llama_cpp_dir, 'gguf-py')
    # Install llama.cpp gguf-py package into the Colab environment (not an isolated pydeps)
    # so the conversion script uses the same well-tested transformers version as Colab.
    run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', gguf_py_dir])
    # Run conversion without -S/isolated mode so it uses Colab's system transformers.
    run_checked([sys.executable, convert_script, merged_dir, '--outfile', GGUF_F16_PATH, '--outtype', 'f16'], cwd=llama_cpp_dir)
    return GGUF_F16_PATH

if os.path.exists(GGUF_DIR):
    shutil.rmtree(GGUF_DIR)
os.makedirs(GGUF_DIR, exist_ok=True)

try:
    clear_unsloth_llama_cpp_cache()
    trainer.model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=QUANT_METHOD)
    gguf_files = sorted(glob.glob(f'{GGUF_DIR}/*.gguf'))
    if not gguf_files:
        raise RuntimeError(f'No GGUF files were created in {GGUF_DIR}')
    FINAL_GGUF_FILE = gguf_files[0]
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
    print('Native GGUF export complete:', FINAL_GGUF_FILE)
except Exception as exc:
    print('Native GGUF export failed, falling back to manual llama.cpp conversion:', exc)
    if os.path.exists(MERGE_DIR):
        shutil.rmtree(MERGE_DIR)
    try:
        trainer.model.save_pretrained_merged(MERGE_DIR, tokenizer, save_method='merged_16bit')
    except RuntimeError as merge_err:
        # Unsloth's post-merge LoRA count check can fire spuriously (# of LoRAs != # of saved modules)
        # after a successful merge — the merge itself produces .safetensors files.
        # Check if merge actually wrote files; if yes, proceed with GGUF conversion.
        merged_files = glob.glob(f'{MERGE_DIR}/*.safetensors') + glob.glob(f'{MERGE_DIR}/*.bin')
        if not merged_files:
            raise RuntimeError(f'Merge failed — no model weight files found in {MERGE_DIR}') from merge_err
        print(f'Merge count-check error ignored — {len(merged_files)} weight file(s) found in {MERGE_DIR}')
    FINAL_GGUF_FILE = manual_llama_cpp_convert(MERGE_DIR)
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)

if not FINAL_GGUF_FILE or not os.path.isfile(FINAL_GGUF_FILE):
    raise RuntimeError('GGUF export failed: no .gguf file was created.')

print('Final GGUF file:', FINAL_GGUF_FILE)
torch.cuda.empty_cache()
gc.collect()

## Cell 9 — Generate model card and optional upload

In [ ]:
MODEL_CARD_PATH = f'{GGUF_DIR}/README.md'
GGUF_FILENAME = globals().get('GGUF_FILENAME') or f"{HF_REPO.split('/')[-1]}-unsloth-{QUANT_METHOD.upper()}.gguf"
SYSTEM_PROMPT_MODELCARD = SYSTEM_PROMPT.replace('"""', '\"\"\"')

model_card_content = f'''---
base_model: {MODEL_NAME}
library_name: unsloth
tags:
- mcp
- weather-sensors
- tool-calling
- gguf
- ministral
---

# Weather Sensors MCP Agent - {MODEL_NAME.split('/')[-1]}

This model was fine-tuned for the Weather Sensors MCP text `tool_call:` contract.
'''

with open(MODEL_CARD_PATH, 'w', encoding='utf-8') as handle:
    handle.write(model_card_content)

print('Model card generated at:', MODEL_CARD_PATH)

# ── Generate Modelfile (required by download-hf-model.sh for Ollama tool support) ──
MODELFILE_PATH = f'{GGUF_DIR}/Modelfile'
modelfile_content = f'''FROM {GGUF_FILENAME}

TEMPLATE """{{{{ if .System }}}}[INST] {{{{ .System }}}}

{{{{ .Prompt }}}} [/INST]{{{{ else }}}}[INST] {{{{ .Prompt }}}} [/INST]{{{{ end }}}}"""

PARAMETER stop "</s>"
PARAMETER num_ctx {MAX_SEQ_LENGTH}
PARAMETER num_thread 4
'''

with open(MODELFILE_PATH, 'w', encoding='utf-8') as handle:
    handle.write(modelfile_content)
print('Modelfile generated at:', MODELFILE_PATH)

UPLOAD_TO_HF = False  # set to True when you want to upload immediately

if UPLOAD_TO_HF:
    import getpass
    from huggingface_hub import HfApi, upload_folder
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('Found HF_TOKEN in Colab Secrets.')
    except Exception:
        print('HF_TOKEN not found in secrets. Please paste your WRITE token.')
        hf_token = getpass.getpass('Hugging Face WRITE Token: ')

    api = HfApi(token=hf_token)
    api.create_repo(repo_id=HF_REPO, exist_ok=True, repo_type='model')
    upload_folder(folder_path=str(GGUF_DIR), repo_id=HF_REPO, repo_type='model', token=hf_token)
    print('Upload complete:', HF_REPO)
else:
    print('Set UPLOAD_TO_HF = True in this cell to upload the exported artifacts.')